In [1]:
import os
import pandas as pd
import cv2
import numpy as np
from moviepy.editor import VideoFileClip, vfx
from tqdm.notebook import tqdm
import json
from datetime import datetime

print(" Libraries imported")

 Libraries imported


In [2]:
# ===============================
# CONFIGURATION
# ===============================
CONFIG = {
    # Input
    'original_csv': 'dataset/dataset.csv',  # Your original dataset
    
    # Output
    'augmented_dir': 'dataset/clean_augmented',
    'output_csv': 'dataset/dataset_clean_augmented.csv',
    'metadata_json': 'dataset/augmentation_metadata.json',
    
    # Augmentation settings
    'brightness_factor': 1.08,  # 8% brighter (you can adjust)
    
    # Processing
    'skip_existing': True,  # Skip if already created (for resume)
}

os.makedirs(CONFIG['augmented_dir'], exist_ok=True)

print("\n Plan:")
print("   For each original video, create:")
print("   1. Original (copy to new structure)")
print("   2. Mirror")
print("   3. Brightness")
print("   4. Mirror + Brightness")
print("\n Result: 755 × 4 = 3,020 videos")



 Plan:
   For each original video, create:
   1. Original (copy to new structure)
   2. Mirror
   3. Brightness
   4. Mirror + Brightness

 Result: 755 × 4 = 3,020 videos


In [3]:
# ===============================
# AUGMENTATION FUNCTIONS
# ===============================

def apply_brightness(frame, factor):
    """Apply brightness adjustment"""
    return cv2.convertScaleAbs(frame, alpha=factor, beta=0)

def augment_video(input_path, output_path, mirror=False, brightness=False):
    """
    Apply augmentations to video
    
    Args:
        input_path: Source video path
        output_path: Destination path
        mirror: Apply horizontal flip
        brightness: Apply brightness adjustment
    """
    try:
        clip = VideoFileClip(input_path)
        original_size = clip.size
        
        # Apply mirror if needed
        if mirror:
            clip = clip.fx(vfx.mirror_x)
        
        # Apply brightness if needed
        if brightness:
            brightness_factor = CONFIG['brightness_factor']
            
            def adjust_brightness(frame):
                return apply_brightness(frame, brightness_factor)
            
            clip = clip.fl_image(adjust_brightness)
        
        # Ensure same resolution
        clip = clip.resize(newsize=original_size)
        
        # Save
        clip.write_videofile(
            output_path, 
            codec="libx264", 
            audio=False,
            verbose=False, 
            logger=None, 
            preset='ultrafast'
        )
        
        clip.close()
        return True, None
        
    except Exception as e:
        return False, str(e)

print(" Augmentation functions ready")

 Augmentation functions ready


In [4]:
# ===============================
# LOAD ORIGINAL DATASET
# ===============================

print(" Loading original dataset...")

df_original = pd.read_csv(CONFIG['original_csv'], encoding='utf-8-sig')
df_original = df_original[df_original['file'].notna()].reset_index(drop=True)

print(f" Loaded {len(df_original)} original videos")
print(f" Gestures: {df_original['gesture'].nunique()}")

# Show distribution
print("\n Original distribution:")
gesture_counts = df_original['gesture'].value_counts()
print(gesture_counts)

print(f"\n Statistics:")
print(f"   Min videos per gesture: {gesture_counts.min()}")
print(f"   Max videos per gesture: {gesture_counts.max()}")
print(f"   Average: {gesture_counts.mean():.1f}")
print(f"   Total: {len(df_original)}")

 Loading original dataset...
 Loaded 755 original videos
 Gestures: 28

 Original distribution:
gesture
السلام وعلیکم                      30
کیا تم بہرے ہو؟                    30
آپ کیسے ہیں؟                      30
ادھر آو!                           30
مبارک ہو                           30
کیا میں اپ کی مدد کر سکتا ہوں؟     30
ایمبولینس کو کال کریں              30
کیسے                               30
کیا آپ پاس یا ناکام ہوئے ہیں؟      30
کیا آپ تیار ہیں؟                   28
میں اشاروں کی زبان سیکھ رہا ہوں    28
آپ کیا کھانا چاہیں گے؟             28
آپ کا نام کیا ہے؟                  28
شکریہ                              28
میں آپ کے ساتھ ہوں                 27
آپ کہاں رہتے ہیں؟                 27
آہِستہ                             25
میں ٹھیک ہوں                       25
نہیں                               25
آپ بہت خوبصورت ہیں۔                25
چھونا مت                           25
خوش رہو                            25
حالات کیسے ہیں؟                    25
میں واش روم جا سکتا ہو

In [5]:
# ===============================
# AUGMENTATION PROCESS
# ===============================
print(" STARTING AUGMENTATION")

augmented_data = []
failed_videos = []
stats = {
    'original': 0,
    'mirror': 0,
    'brightness': 0,
    'mirror_brightness': 0,
    'failed': 0,
    'skipped': 0
}

# Track which original video each augmentation came from (for proper splitting later)
augmentation_groups = {}
group_id = 0

for idx, row in tqdm(df_original.iterrows(), total=len(df_original), desc="Processing videos"):
    
    original_path = row['file']
    gesture = row['gesture']
    gesture_type = row.get('type', 'dynamic')
    hand = row.get('hand', 'both')
    
    # Check if original file exists
    if not os.path.exists(original_path):
        stats['failed'] += 1
        failed_videos.append({
            'file': original_path,
            'reason': 'File not found'
        })
        continue
    
    # Create gesture folder in new structure
    gesture_folder_name = os.path.basename(os.path.dirname(original_path))
    gesture_output_dir = os.path.join(CONFIG['augmented_dir'], gesture_folder_name)
    os.makedirs(gesture_output_dir, exist_ok=True)
    
    # Base filename
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    
    # Track this group
    current_group_id = group_id
    group_id += 1
    
    # ========================================
    # 1. ORIGINAL (just copy to new structure)
    # ========================================
    original_output_name = f"{base_name}.mp4"
    original_output_path = os.path.join(gesture_output_dir, original_output_name)
    
    if CONFIG['skip_existing'] and os.path.exists(original_output_path):
        stats['skipped'] += 1
    else:
        # Copy original
        success, error = augment_video(original_path, original_output_path, 
                                      mirror=False, brightness=False)
        if success:
            stats['original'] += 1
        else:
            stats['failed'] += 1
            failed_videos.append({
                'file': original_path,
                'augmentation': 'original',
                'reason': error
            })
            continue
    
    # Add to dataset
    augmented_data.append({
        'gesture': gesture,
        'file': original_output_path.replace("\\", "/"),
        'type': gesture_type,
        'hand': hand,
        'augmentation': 'none',
        'group_id': current_group_id,
        'source_video': original_path
    })
    
    # ========================================
    # 2. MIRROR
    # ========================================
    mirror_output_name = f"{base_name}_mirror.mp4"
    mirror_output_path = os.path.join(gesture_output_dir, mirror_output_name)
    
    if CONFIG['skip_existing'] and os.path.exists(mirror_output_path):
        stats['skipped'] += 1
    else:
        success, error = augment_video(original_path, mirror_output_path,
                                      mirror=True, brightness=False)
        if success:
            stats['mirror'] += 1
        else:
            stats['failed'] += 1
            failed_videos.append({
                'file': original_path,
                'augmentation': 'mirror',
                'reason': error
            })
            # Don't continue - try other augmentations
    
    if os.path.exists(mirror_output_path):
        augmented_data.append({
            'gesture': gesture,
            'file': mirror_output_path.replace("\\", "/"),
            'type': gesture_type,
            'hand': hand,
            'augmentation': 'mirror',
            'group_id': current_group_id,
            'source_video': original_path
        })
    
    # ========================================
    # 3. BRIGHTNESS
    # ========================================
    brightness_output_name = f"{base_name}_bright.mp4"
    brightness_output_path = os.path.join(gesture_output_dir, brightness_output_name)
    
    if CONFIG['skip_existing'] and os.path.exists(brightness_output_path):
        stats['skipped'] += 1
    else:
        success, error = augment_video(original_path, brightness_output_path,
                                      mirror=False, brightness=True)
        if success:
            stats['brightness'] += 1
        else:
            stats['failed'] += 1
            failed_videos.append({
                'file': original_path,
                'augmentation': 'brightness',
                'reason': error
            })
    
    if os.path.exists(brightness_output_path):
        augmented_data.append({
            'gesture': gesture,
            'file': brightness_output_path.replace("\\", "/"),
            'type': gesture_type,
            'hand': hand,
            'augmentation': 'brightness',
            'group_id': current_group_id,
            'source_video': original_path
        })
    
    # ========================================
    # 4. MIRROR + BRIGHTNESS
    # ========================================
    mirror_bright_output_name = f"{base_name}_mirror_bright.mp4"
    mirror_bright_output_path = os.path.join(gesture_output_dir, mirror_bright_output_name)
    
    if CONFIG['skip_existing'] and os.path.exists(mirror_bright_output_path):
        stats['skipped'] += 1
    else:
        success, error = augment_video(original_path, mirror_bright_output_path,
                                      mirror=True, brightness=True)
        if success:
            stats['mirror_brightness'] += 1
        else:
            stats['failed'] += 1
            failed_videos.append({
                'file': original_path,
                'augmentation': 'mirror_brightness',
                'reason': error
            })
    
    if os.path.exists(mirror_bright_output_path):
        augmented_data.append({
            'gesture': gesture,
            'file': mirror_bright_output_path.replace("\\", "/"),
            'type': gesture_type,
            'hand': hand,
            'augmentation': 'mirror_brightness',
            'group_id': current_group_id,
            'source_video': original_path
        })

print("\n Augmentation complete!")

🚀 STARTING AUGMENTATION


Processing videos:   0%|          | 0/755 [00:00<?, ?it/s]


 Augmentation complete!


In [6]:
# ===============================
# SAVE RESULTS
# ===============================
print(" SAVING RESULTS")

# Save augmented dataset CSV
df_augmented = pd.DataFrame(augmented_data)
df_augmented.to_csv(CONFIG['output_csv'], index=False, encoding='utf-8-sig')
print(f"\n Saved augmented dataset: {CONFIG['output_csv']}")

# Save metadata
metadata = {
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'original_videos': len(df_original),
    'augmented_videos': len(df_augmented),
    'total_groups': group_id,
    'brightness_factor': CONFIG['brightness_factor'],
    'statistics': stats,
    'failed_videos': failed_videos
}

with open(CONFIG['metadata_json'], 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)
print(f" Saved metadata: {CONFIG['metadata_json']}")

# Save failed videos log if any
if failed_videos:
    failed_log = 'dataset/failed_augmentations.json'
    with open(failed_log, 'w', encoding='utf-8') as f:
        json.dump(failed_videos, f, ensure_ascii=False, indent=4)
    print(f"  Saved failed videos log: {failed_log}")

💾 SAVING RESULTS

 Saved augmented dataset: dataset/dataset_clean_augmented.csv
 Saved metadata: dataset/augmentation_metadata.json


In [7]:
# ===============================
#FINAL STATISTICS
# ===============================

print(" AUGMENTATION STATISTICS")

print(f"\n Successfully created:")
print(f"   Original (copied):      {stats['original']:4d}")
print(f"   Mirror:                 {stats['mirror']:4d}")
print(f"   Brightness:             {stats['brightness']:4d}")
print(f"   Mirror + Brightness:    {stats['mirror_brightness']:4d}")
print(f"   ─────────────────────────────")
print(f"   Total augmented:        {len(df_augmented):4d}")

if stats['skipped'] > 0:
    print(f"\n  Skipped (already exist): {stats['skipped']}")

if stats['failed'] > 0:
    print(f"\n Failed:                  {stats['failed']}")

print(f"\n Dataset growth:")
print(f"   Original dataset:       {len(df_original):4d} videos")
print(f"   Augmented dataset:      {len(df_augmented):4d} videos")
print(f"   Multiplication factor:  {len(df_augmented)/len(df_original):.1f}x")

# Per-gesture statistics
print(f"\n Per-gesture breakdown:")
augmented_per_gesture = df_augmented['gesture'].value_counts().sort_index()

comparison = pd.DataFrame({
    'Original': gesture_counts.sort_index(),
    'Augmented': augmented_per_gesture,
    'Multiplier': (augmented_per_gesture / gesture_counts.sort_index()).round(1)
})

print(comparison)

# Augmentation type distribution
print(f"\n Augmentation type distribution:")
aug_type_dist = df_augmented['augmentation'].value_counts()
print(aug_type_dist)

print(" AUGMENTATION COMPLETE!")

print(f"\n Output files:")
print(f"   ├─ Augmented videos:  {CONFIG['augmented_dir']}/")
print(f"   ├─ Dataset CSV:       {CONFIG['output_csv']}")
print(f"   └─ Metadata:          {CONFIG['metadata_json']}")


📊 AUGMENTATION STATISTICS

 Successfully created:
   Original (copied):       755
   Mirror:                  755
   Brightness:              755
   Mirror + Brightness:     755
   ─────────────────────────────
   Total augmented:        3020

 Dataset growth:
   Original dataset:        755 videos
   Augmented dataset:      3020 videos
   Multiplication factor:  4.0x

 Per-gesture breakdown:
                                 Original  Augmented  Multiplier
gesture                                                         
آپ بہت خوبصورت ہیں۔                    25        100         4.0
آپ کا نام کیا ہے؟                      28        112         4.0
آپ کیا کھانا چاہیں گے؟                 28        112         4.0
آہِستہ                                 25        100         4.0
ادھر آو!                               30        120         4.0
السلام وعلیکم                          30        120         4.0
آپ کہاں رہتے ہیں؟                     27        108         4.0
آپ کیسے ہیں؟      

In [8]:
# ===============================
# VERIFY AUGMENTATIONS
# ===============================

print("\n VERIFICATION")

# Check a random gesture folder
sample_gesture = df_augmented['gesture'].iloc[0]
sample_files = df_augmented[df_augmented['gesture'] == sample_gesture].head(4)

print(f"\n Sample: {sample_gesture}")
print(f"   Files created from first original video:")

for _, row in sample_files.iterrows():
    file_path = row['file']
    exists = "" if os.path.exists(file_path) else ""
    size_mb = os.path.getsize(file_path) / (1024*1024) if os.path.exists(file_path) else 0
    print(f"   {exists} {row['augmentation']:20s} | {size_mb:.2f} MB | {os.path.basename(file_path)}")

# Check folder structure
print(f"\n Created gesture folders:")
gesture_folders = sorted([d for d in os.listdir(CONFIG['augmented_dir']) 
                         if os.path.isdir(os.path.join(CONFIG['augmented_dir'], d))])

print(f"   Total folders: {len(gesture_folders)}")
for folder in gesture_folders[:5]:  # Show first 5
    folder_path = os.path.join(CONFIG['augmented_dir'], folder)
    num_files = len([f for f in os.listdir(folder_path) if f.endswith('.mp4')])
    print(f"    {folder}: {num_files} videos")

if len(gesture_folders) > 5:
    print(f"   ... and {len(gesture_folders) - 5} more folders")

print("\n Verification complete!")


🔍 VERIFICATION

 Sample: السلام وعلیکم
   Files created from first original video:
    none                 | 0.48 MB | vid_000.mp4
    mirror               | 0.48 MB | vid_000_mirror.mp4
    brightness           | 0.52 MB | vid_000_bright.mp4
    mirror_brightness    | 0.52 MB | vid_000_mirror_bright.mp4

 Created gesture folders:
   Total folders: 28
    آپ بہت خوبصورت ہیں۔: 100 videos
    آپ کا نام کیا ہے؟: 112 videos
    آپ کیا کھانا چاہیں گے؟: 112 videos
    آہِستہ: 100 videos
    ادھر آو!: 120 videos
   ... and 23 more folders

 Verification complete!
